# Weather API Connection and Pipeline

This notebook shows how the project fetches live weather data from Open-Meteo, saves the raw JSON, and prepares it for analysis.

In [ ]:
import json
import os
from pathlib import Path

import requests

# Clear proxy variables that may block outbound API access in this environment
for key in ['HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'http_proxy', 'https_proxy', 'all_proxy']:
    os.environ.pop(key, None)

url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude': 40.7128,
    'longitude': -74.0060,
    'current': 'temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,weather_code,wind_speed_10m',
    'timezone': 'America/New_York',
}

response = requests.get(url, params=params, timeout=30, proxies={'http': None, 'https': None})
response.raise_for_status()
data = response.json()

print('status:', response.status_code)
print('keys:', list(data.keys())[:10])
print('current temp:', data['current'].get('temperature_2m'))

raw_dir = Path('data/raw')
raw_dir.mkdir(exist_ok=True, parents=True)
raw_path = raw_dir / 'new_york_current.json'
raw_path.write_text(json.dumps(data, indent=2), encoding='utf-8')
print('saved to:', raw_path)

## Historical weather fetch

This section fetches historical daily weather records and saves them as a second raw JSON file.

In [ ]:
from datetime import date, timedelta

end_date = date.today().isoformat()
start_date = (date.today() - timedelta(days=365)).isoformat()

historical_url = 'https://archive-api.open-meteo.com/v1/archive'
historical_params = {
    'latitude': 40.7128,
    'longitude': -74.0060,
    'start_date': start_date,
    'end_date': end_date,
    'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum,relative_humidity_2m_mean,wind_speed_10m_max',
    'timezone': 'America/New_York',
}

historical_response = requests.get(historical_url, params=historical_params, timeout=60, proxies={'http': None, 'https': None})
historical_response.raise_for_status()
historical_data = historical_response.json()

historical_path = raw_dir / 'new_york_historical.json'
historical_path.write_text(json.dumps(historical_data, indent=2), encoding='utf-8')
print('historical days:', len(historical_data.get('daily', {}).get('time', [])))
print('saved to:', historical_path)